# Notebook 3: Improved Encoding with BLOSUM62

One-hot encoding treats every amino acid as equally different from every other.
This is biochemically wrong: leucine and isoleucine are nearly interchangeable,
while alanine and tryptophan almost never substitute for each other.

BLOSUM62 is a substitution scoring matrix derived from aligned blocks of
evolutionarily related proteins. Each amino acid is represented as a 20-dimensional
vector of substitution scores, encoding its biochemical similarity to all other amino acids.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from Bio.Align import substitution_matrices

df = pd.read_csv("protein_sequences_ss_3.csv")

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}


In [ ]:
# BLOSUM62 encoding
blosum = substitution_matrices.load("BLOSUM62")

# Pre-compute a 20×20 matrix for fast lookup
blosum_matrix = np.array([
    [float(blosum[(aa1, aa2)]) for aa2 in AMINO_ACIDS]
    for aa1 in AMINO_ACIDS
])

def blosum_encode(aa):
    """Return a 20-dim BLOSUM62 row for amino acid aa, or zeros for unknowns."""
    if aa not in aa_to_index:
        return np.zeros(20)
    return blosum_matrix[aa_to_index[aa]]


print("BLOSUM62 vector for Alanine (A):")
print(blosum_encode("A"))


BLOSUM62 vector for Alanine (A):
[ 4.  0. -2. -1. -2.  0. -2. -1. -1. -1. -1. -2. -1. -1. -1.  1.  0.  0.
 -3. -2.]


In [ ]:
# Protein-level split 
protein_ids = df.index.tolist()
train_ids, test_ids = train_test_split(protein_ids, test_size=0.2, random_state=42)
train_df = df.iloc[train_ids].reset_index(drop=True)
test_df  = df.iloc[test_ids].reset_index(drop=True)


In [ ]:
# Feature extraction with BLOSUM62
WINDOW_SIZE = 15
HALF        = WINDOW_SIZE // 2

def build_features(subset_df):
    X, y = [], []
    for _, row in subset_df.iterrows():
        seq = row["sequence"]
        ss  = row["secondary_structure"]
        padded = "X" * HALF + seq + "X" * HALF
        for i in range(HALF, len(padded) - HALF):
            window = padded[i - HALF : i + HALF + 1]
            X.append(np.concatenate([blosum_encode(aa) for aa in window]))
            y.append(ss[i - HALF])
    return np.array(X), np.array(y)

X_train, y_train = build_features(train_df)
X_test,  y_test  = build_features(test_df)

print(f"Feature shape: {X_train.shape}  (same size as one-hot, richer values)")


Feature shape: (138806, 300)  (same size as one-hot, richer values)


In [ ]:
# Fit label encoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


In [ ]:
# Random Forest + BLOSUM62
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train_enc)
rf_pred = rf.predict(X_test)
print("Random Forest + BLOSUM62")
print(classification_report(y_test_enc, rf_pred, target_names=["H","E","C"]))


Random Forest + BLOSUM62
              precision    recall  f1-score   support

           H       0.65      0.73      0.69     12974
           E       0.68      0.41      0.51      6537
           C       0.69      0.75      0.72     13698

    accuracy                           0.67     33209
   macro avg       0.68      0.63      0.64     33209
weighted avg       0.67      0.67      0.67     33209



In [ ]:
# XGBoost + BLOSUM62 
xgb = XGBClassifier(n_estimators=200, max_depth=6, eval_metric="mlogloss",
                    random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train_enc)

xgb_pred     = le.inverse_transform(xgb.predict(X_test))
y_test_labels = le.inverse_transform(y_test_enc)

print("XGBoost + BLOSUM62")
print(classification_report(y_test_labels, xgb_pred))
print("\nBLOSUM62 improved E recall from 0.42 → 0.50 (XGBoost).")
print("Biochemical similarity information helps the model generalise across amino acid families.")


XGBoost + BLOSUM62
              precision    recall  f1-score   support

           C       0.66      0.72      0.69     12974
           E       0.64      0.50      0.56      6537
           H       0.73      0.75      0.74     13698

    accuracy                           0.69     33209
   macro avg       0.68      0.66      0.66     33209
weighted avg       0.69      0.69      0.69     33209


BLOSUM62 improved E recall from 0.42 → 0.50 (XGBoost).
Biochemical similarity information helps the model generalise across amino acid families.
